In [4]:
from sympy import symbols, oo
from sympy.integrals.transforms import z_transform

n, z = symbols('n z')

# Calcula a transformada Z de 1^n
resultado = z_transform(1**n, n, z)
print(resultado)  # Saída esperada: z/(z - 1)


ImportError: cannot import name 'z_transform' from 'sympy.integrals.transforms' (d:\arqM2100\Ensino\aulas_PDS\.venv\Lib\site-packages\sympy\integrals\transforms.py)

In [5]:
from sympy import symbols, Sum, oo, simplify, pi

def z_transform(expr, n_sym, z_sym):
    """Calcula a transformada Z unilateral usando a definição de somatório."""
    # Retorna o somatório de n=0 até o infinito de: expr * z^(-n)
    return Sum(expr * (z_sym ** -n_sym), (n_sym, 0, oo)).doit()

# 1. Definindo as variáveis simbólicas
n, z = symbols('n z')
a = symbols('a', constant=True)

# 2. Exemplo: Transformada Z de a^n (exponencial)
# Expressão: a**n
resultado_exp = z_transform(a**n, n, z)
print("Transformada de a^n:", simplify(resultado_exp))
# Saída esperada: z/(z - a)

# 3. Exemplo: Transformada Z de 1 (Degrau unitário, onde a=1)
resultado_degrau = z_transform(1, n, z)
print("Transformada do Degrau:", simplify(resultado_degrau))
# Saída esperada: z/(z - 1)


Transformada de a^n: Piecewise((z/(-a + z), Abs(a/z) < 1), (Sum(a**n/z**n, (n, 0, oo)), True))
Transformada do Degrau: Piecewise((z/(z - 1), (z > 1) | (z < -1)), (Sum(z**(-n), (n, 0, oo)), True))


In [6]:
from sympy import symbols, Eq, solve, simplify

# 1. Definindo as variáveis simbólicas
z = symbols('z')
Y = symbols('Y')  # Representando Y(z)

# 2. Definindo a Transformada Z da entrada x[n] = u[n] (Degrau Unitário)
X_z = z / (z - 1)

# 3. Substituindo a equação de diferenças pelo seu equivalente no domínio Z
# y[n] -> Y
# -0.5*y[n-1] -> -0.5 * Y * z**(-1)
# +0.06*y[n-2] -> +0.06 * Y * z**(-2)
# x[n] -> X_z
equacao_z = Eq(Y - 0.5 * Y * z**(-1) + 0.06 * Y * z**(-2), X_z)

# 4. Isolando Y(z) na equação
solucao_Y = solve(equacao_z, Y)[0]

# 5. Simplificando a expressão para o formato padrão de frações polinomiais
Y_z_formatado = simplify(solucao_Y)

print("Y(z) =", Y_z_formatado)


Y(z) = 50.0*z**3/(50.0*z**3 - 75.0*z**2 + 28.0*z - 3.0)


In [7]:
from sympy import symbols, Eq, solve, simplify, apart, InverseLaplaceTransform

# 1. Definindo as variáveis simbólicas (n deve ser inteiro e positivo)
z = symbols('z')
n = symbols('n', integer=True, positive=True)
Y = symbols('Y')

# 2. Entrada x[n] = u[n] -> X(z) = z / (z - 1)
X_z = z / (z - 1)

# 3. Montando a equação de diferenças no domínio Z
equacao_z = Eq(Y - 0.5 * Y * z**(-1) + 0.06 * Y * z**(-2), X_z)

# 4. Isolando Y(z)
Y_z = solve(equacao_z, Y)[0]

# 5. Método das Frações Parciais para Transformada Z:
# Expandimos Y(z)/z em fractions parciais
Y_z_sobre_z = simplify(Y_z / z)
fracoes_parciais = apart(Y_z_sobre_z)

print("Expansão de Y(z)/z em frações parciais:")
print(fracoes_parciais)
# Isso vai gerar algo como: A/(z-1) + B/(z-0.3) + C/(z-0.2)

# 6. Reconstruindo Y(z) multiplicando cada termo de volta por z
# E aplicando a transformada inversa: z/(z-a) -> a^n
y_n = 0
for termo in fracoes_parciais.args:
    # Multiplica o termo por z para voltar ao formato padrão z/(z-a)
    termo_z = termo * z
    
    # Identifica o polo 'a' no denominador (z - a)
    # Procuramos o denominador do termo atual
    den = termo.as_numer_denom()[1]
    
    # Se o denominador for (z - 1), o polo é 1. Se for (z - 0.3), o polo é 0.3, etc.
    # Uma forma simples de achar o polo é resolver: den = 0
    polo = solve(den, z)[0]
    
    # O coeficiente do termo z/(z-a)
    coeficiente = simplify(termo_z / (z / (z - polo)))
    
    # Adiciona o termo correspondente em n: coeficiente * (polo)^n
    y_n += coeficiente * (polo**n)

print("\n--------------------------------------------------")
print("Resposta no tempo y[n] para n >= 0:")
print(simplify(y_n))


Expansão de Y(z)/z em frações parciais:
0.5/(1.0*z - 0.2) + 1.78571428571429/(z - 1) - 1.28571428571429/(1.0*z - 0.3)

--------------------------------------------------
Resposta no tempo y[n] para n >= 0:
0.5*0.2**n - 1.28571428571429*0.3**n + 1.78571428571429


In [9]:
25/14

1.7857142857142858

In [9]:
import numpy as np
import scipy.signal as sig

# Coeficientes do numerador e denominador
num = [1, 0, 0]  # z^3
den = [1, -0.5, 0.06]  # z^2 - 0.5z + 0.06

den = np.polymul([1,-1], den)


# Cálculo de resíduos, polos e termo direto (residue)
residuos, polos, direto = sig.residue(num, den)

# Cálculo explícito de polos e zeros com raízes dos polinômios
todos_polos = np.roots(den)


print('Resíduos:', residuos)
print('Polos:', polos)



Resíduos: [ 0.5        -1.28571429  1.78571429]
Polos: [0.2 0.3 1. ]


In [7]:
direto

array([], dtype=float64)